## Evaluation of poetry_analysis on test sets

* Manually annotated test sets are placed in the `tests/evaluation_data` directory for end rhymes, alliteration, anaphora and epiphora. 
* For each literary device, run all relevant functions to get a comprehensive result. Anaphora and epiphora can work on different text levels: verse line, stanza, poem. Alliteration and end rhymes can be detected from text or phonetic transcriptions, and alliteration can be detected with or without intervening words (typically function words). 
* Report a summary of the results with precision, recall, and accuracy/harmonic mean (F1-score).

In [1]:
import pandas as pd


def report_results(true_positive: pd.Series, false_positive: pd.Series, false_negative: pd.Series):
    precision = true_positive.sum() / (true_positive.sum() + false_positive.sum())

    recall = true_positive.sum() / (true_positive.sum() + false_negative.sum())

    f1_score = (2 * true_positive.sum()) / (2 * true_positive.sum() + false_positive.sum() + false_negative.sum())

    print(f"Precision: {precision * 100:2f}%")
    print(f"Recall: {recall * 100:2f}%")
    print(f"F1 score: {f1_score * 100:2f}%")


def calculate_metrics(data: pd.DataFrame, ref_col, hyp_col, save_errors_filestem: str | None = None):
    """Calculate precision, recall and F1 score for a dataframe with a reference and a hypothesis."""
    true_positive = data[ref_col].notna() & data[hyp_col].notna()
    false_positive = data[ref_col].isna() & data[hyp_col].notna()
    false_negative = data[ref_col].notna() & data[hyp_col].isna()

    # Save errors to disk for further analysis
    if save_errors_filestem is not None:
        data[false_positive][["verse_id", hyp_col]].to_csv(f"{save_errors_filestem}_false_positive.csv")

        data[false_negative][["verse_id", ref_col]].to_csv(f"{save_errors_filestem}_false_negative.csv")

    report_results(true_positive, false_positive, false_negative)

## End rhymes



### Multilabel classification

For multilabel classification algorithms, we compute accuracy as number of correct labels divided by number of wrong classifications. The label is called "rhyme_tag" in this testset. 

Evaluate overall accuracy of rhyme tag (multilabel classification) on:
- Orthographic texts
- Phonemic transcriptions

In [2]:
import pandas as pd

rhyme_testset = pd.read_excel("evaluation_data/norn_poems_testset_rhyme.xlsx")

correct_text = (rhyme_testset.rhyme_tag_text == rhyme_testset.rhyme_tag_gold).sum()
correct_syll = (rhyme_testset.rhyme_tag_syll == rhyme_testset.rhyme_tag_gold).sum()
all_labels = rhyme_testset.rhyme_tag_gold.count()

In [19]:
# Count occurrences of rhymes among verse lines
end_rhyme_frequency = (rhyme_testset.rhyme_score_gold > 0).sum() / rhyme_testset.rhyme_tag_gold.count()

print(f"Number of lines that rhyme with another line {end_rhyme_frequency * 100}%")

Number of lines that rhyme with another line 37.19531497309275%


### Overall accuracy of end rhyme tags in orthographic texts 

In [3]:
accuracy_text = correct_text / all_labels

print(f"the rhyme tagger annotates end rhymes on orthographic text with an accuracy of {accuracy_text * 100:.2f}%")

the rhyme tagger annotates end rhymes on orthographic text with an accuracy of 78.41%


### Overall accuracy of end rhyme tags in phonemic transcriptions 

The transcriptions have been predicted with a G2P-model for modern Norwegian bokmål and the eastern Norwegian dialect: 


Språkbanken, the National Library of Norway. Grapheme-to-Phoneme Models for Norwegian. Python. V. 2. Oslo, Released 9 February 2024. https://www.nb.no/sprakbanken/ressurskatalog/oai-nb-no-sbr-93/.



In [ ]:
accuracy_syll = correct_syll / all_labels

print(
    f"the rhyme tagger annotates end rhymes on phonemic transcriptions with an accuracy of {accuracy_syll * 100:.2f}%"
)

### Binary detection of rhyme

When we switch from evaluating whether the algorithm got the rhyme tags correct, to evaluating whether it has correctly detected that a line rhymes with another line, the accuracy increases, as expected. The labels are incrementally assigned starting from the beginning of the alphabet, so the pattern is skewed if two lines are mislabelled early in a long stanza.


In [ ]:
correct_text_score = (rhyme_testset["rhyme_score_text"] == rhyme_testset["rhyme_score_gold"]).sum()
correct_syll_score = (rhyme_testset["rhyme_score_syll"] == rhyme_testset["rhyme_score_gold"]).sum()
all_scores = rhyme_testset["rhyme_score_gold"].count()

### Overall accuracy of end rhyme detection in orthographic texts 

In [ ]:
accuracy_text = correct_text_score / all_scores

print(
    f"the rhyme tagger annotates (binary) end rhymes on orthographic text with an accuracy of {accuracy_text * 100:.2f}%"
)

### Overall accuracy of end rhyme detection in phonemic transcriptions

In [ ]:
accuracy_syll = correct_syll_score / all_scores

print(
    f"the rhyme tagger annotates (binary) end rhymes on phonemic transcriptions with an accuracy of {accuracy_syll * 100:.2f}%"
)

In [ ]:
## Error analysis: write false negatives for rhyme detection to csv

# CONFIGURE: uncomment the modality you want error analysis for, and comment out the other one
# modality = "text"
modality = "syll"

condition = (rhyme_testset["rhyme_score_gold"] > 0) & (rhyme_testset[f"rhyme_score_{modality}"] == 0)

rhyme_testset[condition][["verse_id", "rhyme_tag_gold", "text"]].to_csv(f"false_negatives_{modality}_rhyme.csv")

In [ ]:
## Error analysis: write false positives for rhyme detection to csv

# CONFIGURE: uncomment the modality you want error analysis for, and comment out the other one
# modality = "text"
modality = "syll"

condition = (rhyme_testset["rhyme_score_gold"] == 0) & (rhyme_testset[f"rhyme_score_{modality}"] > 0)

rhyme_testset[condition][["verse_id", "rhyme_tag_gold", "text"]].to_csv(f"false_positive_{modality}_rhyme.csv")

## Alliteration

Evaluate precision / recall on the following alliteration types: 
- word-initial letter in text
- vowels 
- consonants
<!--  
- word-initial sound in transcriptions 
- word-initial letter cluster (hv / sj / kj)
-->

In [20]:
import pandas as pd

from poetry_analysis.alliteration import find_line_alliterations, is_vowel

alllit_testset = pd.read_excel("evaluation_data/norn_poems_testset_alliteration.xlsx")

alllit_testset["alliteration_gold"] = alllit_testset.alliteration_gold.str.strip()

# Add columns to filter consonants and vowel patterns from the gold standard
alllit_testset["is_vowel"] = alllit_testset.alliteration_symbol_gold.apply(
    lambda x: any(is_vowel(c) for c in str(x).split(";") if isinstance(x, str))
)

alllit_testset["is_consonant"] = alllit_testset.alliteration_symbol_gold.apply(
    lambda x: any(not is_vowel(c) for c in x.split(";")) if isinstance(x, str) else False
)


# Extract words with alliterating and reformat to match the gold standard
alllit_testset["alliteration_prediction"] = alllit_testset.text.apply(
    find_line_alliterations,
).apply(lambda line: "; ".join(" ".join(words) for words in line) if line else None)
# Vowels only
alllit_testset["alliteration_pred_vowel"] = alllit_testset.text.apply(
    find_line_alliterations, letter_type="vowel"
).apply(lambda line: "; ".join(" ".join(words) for words in line) if line else None)

# Consonants only
alllit_testset["alliteration_pred_consonant"] = alllit_testset.text.apply(
    find_line_alliterations, letter_type="consonant"
).apply(lambda line: "; ".join(" ".join(words) for words in line) if line else None)


# Extract words with alliteration patterns, allowing more words to appear between the alliterating words
# ["og", "mod", "i", "er", "mot", "med", "som", "ved", "men", "om", "en", "af", "av", "at", "fra", "på"]
conjunctions = ["og", "samt", "eller", "men", "for", "så", "saa"]
subjunctions = ["at", "hvis", "å", "aa", "som", "der"]
prepositions = [
    "i",
    "til",
    "fra",
    "med",
    "mæ",
    "av",
    "af",
    "på",
    "paa",
    "under",
    "over",
    "yvi",
    "yver",
    "foran",
    "bak",
    "mellom",
    "hos",
    "før",
    "etter",
    "efter",
    "om",
    "ved",
    "mot",
    "mod",
]
determinatives = [
    "en",
    "et",
    "ein",
    "ett",
    "det",
    "den",
    "dette",
    "denne",
    "de",
    "disse",
    "min",
    "mine",
    "mitt",
    "mit",
    "din",
    "dine",
    "ditt",
    "dit",
    "hans",
    "hennes",
    "deres",
    "våre",
]
pronouns = [
    "jeg",
    "eg",
    "meg",
    "du",
    "deg",
    "han",
    "ham",
    "hun",
    "ho",
    "hen",
    "henne",
    "vi",
    "oss",
    "dere",
    "dykk",
    "dokker",
    "dei",
    "dem",
]
alllit_testset["alliteration_pred_extra_words"] = alllit_testset.text.apply(
    find_line_alliterations,
    allowed_intervening_words=conjunctions + subjunctions + prepositions + determinatives,  # + pronouns
).apply(lambda line: "; ".join(" ".join(words) for words in line) if line else None)

In [ ]:
alliteration_frequency = alllit_testset.alliteration_gold.count() / len(alllit_testset)


print(
    f"Number of verse lines where an alliteration occurs: {alllit_testset.alliteration_gold.count()} ({alliteration_frequency * 100}%)"
)

Number of verse lines where an alliteration occurs: 484(15.587761674718195%)


In [ ]:
# EVALUATE

print("Any letter type:")
calculate_metrics(
    alllit_testset, "alliteration_gold", "alliteration_prediction", save_errors_filestem="alliteration_all"
)
print("\n")

print("Vowels: ")
true_positive = alllit_testset["is_vowel"] & alllit_testset["alliteration_pred_vowel"].notna()
false_positive = ~alllit_testset["is_vowel"] & alllit_testset["alliteration_pred_vowel"].notna()
false_negative = alllit_testset["is_vowel"] & alllit_testset["alliteration_pred_vowel"].isna()

# Save incorrect predictions for error analysis
alllit_testset[false_positive][["verse_id", "alliteration_pred_vowel"]].to_csv("alliteration_vowels_false_positive.csv")
alllit_testset[false_negative][["verse_id", "alliteration_gold"]].to_csv("alliteration_vowels_false_negative.csv")

report_results(true_positive, false_positive, false_negative)

print("\n")

print("Consonants: ")
true_positive = alllit_testset["is_consonant"] & alllit_testset["is_consonant"].notna()
false_positive = ~alllit_testset["is_consonant"] & alllit_testset["alliteration_pred_consonant"].notna()
false_negative = alllit_testset["is_consonant"] & alllit_testset["alliteration_pred_consonant"].isna()

# Save incorrect predictions for error analysis
alllit_testset[false_positive][["verse_id", "alliteration_pred_consonant"]].to_csv(
    "alliteration_consonants_false_positive.csv"
)
alllit_testset[false_negative][["verse_id", "alliteration_gold"]].to_csv("alliteration_consonants_false_negative.csv")

report_results(true_positive, false_positive, false_negative)

print("\n")


print("Extra words allowed between: ")
calculate_metrics(
    alllit_testset,
    "alliteration_gold",
    "alliteration_pred_extra_words",
    save_errors_filestem="alliteration_extra_words",
)

## Anaphora

Evaluate precision / recall on the following categories of anaphoric patterns:

- [Line](#line-anaphora): Line-initial word(s) repeated within same line
- [Stanza](#stanza-anaphora): line-initial word(s) repeated within same stanza
- [Across](#anaphora-across-stanzas): Stanza-initial line or partial line repeated across stanzas 
 

In [74]:
import pandas as pd

anaphora_testset = pd.read_excel("evaluation_data/norn_poems_testset_anaphora.xlsx")

anaphora_testset["anaphora_gold"] = anaphora_testset.anaphora_gold.str.strip()

In [75]:
anaphora_frequency = anaphora_testset.anaphora_gold.count() / len(anaphora_testset)

print(f"Number of lines with anaphora: {anaphora_testset.anaphora_gold.count()} ({anaphora_frequency * 100} %)")

Number of lines with anaphora: 370 (11.916264090177133 %)


### Line anaphora

In [76]:
from poetry_analysis.anaphora import extract_line_anaphora

# PREDICT: Repeated line-initial phrases
anaphora_testset["anaphora_line_prediction"] = anaphora_testset.text.apply(
    lambda x: extract_line_anaphora(x).get("phrase", None)
)

# EVALUATE: Count true positives, true negatives, false positives and false negatives
true_positive = (anaphora_testset.anaphora_line == "x") & (
    anaphora_testset.anaphora_gold == anaphora_testset.anaphora_line_prediction
)

false_positive = anaphora_testset.anaphora_line_prediction.notna() & (
    anaphora_testset.anaphora_gold != anaphora_testset.anaphora_line_prediction
)

true_negative = anaphora_testset.anaphora_gold.isna() & anaphora_testset.anaphora_line_prediction.isna()

false_negative = (anaphora_testset.anaphora_line == "x") & (anaphora_testset.anaphora_line_prediction.isna())

# Error analysis
anaphora_testset[false_positive][["verse_id", "anaphora_line_prediction"]].to_csv("anaphora_line_false_positive.csv")
anaphora_testset[false_negative][["verse_id", "anaphora_gold"]].to_csv("anaphora_line_false_negative.csv")


report_results(true_positive, false_positive, false_negative)

Precision: 37.588652%
Recall: 100.000000%
F1 score: 54.639175%


### Stanza anaphora

In [77]:
# PREDICT
from poetry_analysis.anaphora import extract_anaphora

# group lines in the testset into stanzas
anaphora_testset[["poem_id", "stanza_id", "versenumber"]] = anaphora_testset["verse_id"].str.split("_", expand=True)

grouped_texts = anaphora_testset.groupby(["poem_id", "stanza_id"])["text"].agg(list)

# Extract line-initial phrases that are repeated within a stanza
results = grouped_texts.apply(extract_anaphora)

# Flatten `results` into a lookup, on the string verse_id -> overlap
pred_map = {}
for (poem_id, stanza_id), stanza_dict in results.items():  # type: ignore[union-attr]
    if not isinstance(stanza_dict, dict):
        continue
    for verse_num, payload in stanza_dict.items():
        overlap = payload.get("overlap") if isinstance(payload, dict) else None
        pred_map[f"{poem_id}_{stanza_id}_v{verse_num}"] = overlap

# Map predictions back to each original row
anaphora_testset["anaphora_stanza_prediction"] = anaphora_testset.verse_id.apply(lambda v: pred_map.get(v))

anaphora_testset = anaphora_testset.drop(["poem_id", "stanza_id", "versenumber"], axis=1)
# Optional: quick check
# anaphora_testset[["verse_id", "text", "anaphora_gold", "anaphora_stanza_prediction"]].head()

In [78]:
# EVALUATE
# Count true positives, true negatives, false positives and false negatives
true_positive = (anaphora_testset.anaphora_stanza == "x") & (
    anaphora_testset.anaphora_gold == anaphora_testset.anaphora_stanza_prediction
)

false_positive = anaphora_testset.anaphora_stanza_prediction.notna() & (
    anaphora_testset.anaphora_gold != anaphora_testset.anaphora_stanza_prediction
)

true_negative = anaphora_testset.anaphora_gold.isna() & anaphora_testset.anaphora_stanza_prediction.isna()

false_negative = (anaphora_testset.anaphora_stanza == "x") & (anaphora_testset.anaphora_stanza_prediction.isna())

## Error analysis
anaphora_testset[false_positive][["verse_id", "anaphora_stanza_prediction"]].to_csv(
    "anaphora_stanza_false_positive.csv"
)

anaphora_testset[false_negative][["verse_id", "anaphora_gold"]].to_csv("anaphora_stanza_false_negative.csv")

report_results(true_positive, false_positive, false_negative)

Precision: 97.826087%
Recall: 68.181818%
F1 score: 80.357143%


### Anaphora across stanzas

Annotate stanza initial lines repeating across stanzas


In [79]:
from poetry_analysis.anaphora import extract_anaphora

# group lines in the testset into lists of stanzas
stanza_texts = (
    anaphora_testset.assign(
        poem_id=anaphora_testset["verse_id"].str.extract(r"p(\d+)").astype(int),
        _stanza_num=anaphora_testset["verse_id"].str.extract(r"s(\d+)").astype(int),
        _verse_num=anaphora_testset["verse_id"].str.extract(r"v(\d+)").astype(int),
    )
    .sort_values(["poem_id", "_stanza_num", "_verse_num"])
    .groupby(["poem_id", "_stanza_num"], sort=False)["text"]
    .apply("\n".join)
)

grouped_texts = stanza_texts.groupby(level=0, sort=False).apply(list)

# extract stanza-initial lines that are repeated in several stanzas
results = grouped_texts.apply(extract_anaphora)


# Flatten `results` into a lookup, on the string verse_id -> overlap
pred_map = {}
for poem_id, stanza_dict in results.items():  # type: ignore[union-attr]
    if not isinstance(stanza_dict, dict):
        continue
    for stanza_num, payload in stanza_dict.items():
        overlap = payload.get("overlap") if isinstance(payload, dict) else None
        pred_map[f"p{poem_id}_s{stanza_num}_v0"] = overlap

# Map predictions back to each original row
anaphora_testset["anaphora_across_prediction"] = anaphora_testset.verse_id.apply(lambda v: pred_map.get(v))

# Optional: quick check
anaphora_testset[["verse_id", "text", "anaphora_gold", "anaphora_across_prediction"]].head()

,verse_id,text,anaphora_gold,anaphora_across_prediction
0,p20_s0_v0,Tvende floder flyder om helvedes hegn.,NaN,None
1,p20_s0_v1,Den ene hvirvler en sydende strom,NaN,None
2,p20_s0_v2,med glans af smeltet metal.,NaN,None
3,p20_s0_v3,"Den anden vælter sorte,",den,None
4,p20_s0_v4,iskolde vande.,NaN,None


In [80]:
# EVALUATE
# Count true positives, true negatives, false positives and false negatives
true_positive = (anaphora_testset.anaphora_across == "x") & (
    anaphora_testset.anaphora_gold == anaphora_testset.anaphora_across_prediction
)

false_positive = anaphora_testset.anaphora_across_prediction.notna() & (
    anaphora_testset.anaphora_gold != anaphora_testset.anaphora_across_prediction
)

true_negative = anaphora_testset.anaphora_gold.isna() & anaphora_testset.anaphora_across_prediction.isna()

false_negative = (anaphora_testset.anaphora_across == "x") & (anaphora_testset.anaphora_across_prediction.isna())

## Error analysis
anaphora_testset[false_positive][["verse_id", "anaphora_across_prediction"]].to_csv(
    "anaphora_across_false_positive.csv"
)

anaphora_testset[false_negative][["verse_id", "anaphora_gold"]].to_csv("anaphora_across_false_negative.csv")

report_results(true_positive, false_positive, false_negative)

Precision: 57.142857%
Recall: 66.666667%
F1 score: 61.538462%


## Epiphora 

Evaluate precision / recall on the following categories of epiphoric patterns (or epistrophes):

- [Line](#line-epiphora): Line-final word(s) repeated within same line
- [Stanza](#stanza-epiphora): line-final word(s) repeated within same stanza
- [Across](#epiphora-across-stanzas): Stanza-final line or partial line repeated across stanzas 

In [ ]:
import pandas as pd

epiphora_testset = pd.read_excel("evaluation_data/norn_poems_testset_epiphora.xlsx")

# epiphora_testset["epiphora_gold"] = epiphora_testset.epiphora_gold.str.strip()

### Line epiphora


In [ ]:
from poetry_analysis.epiphora import extract_line_epiphora

# PREDICT: Repeated line-initial phrases
epiphora_testset["epiphora_line_prediction"] = epiphora_testset.text.apply(
    lambda x: extract_line_epiphora(x).get("phrase", None)
)

# EVALUATE: Count true positives, true negatives, false positives and false negatives
true_positive = (
    (epiphora_testset.epiphora_line == "x")
    & (epiphora_testset.epiphora_gold == epiphora_testset.epiphora_line_prediction)
).sum()

false_positive = (
    epiphora_testset.epiphora_line_prediction.notna()
    & (epiphora_testset.epiphora_gold != epiphora_testset.epiphora_line_prediction)
).sum()

true_negative = (epiphora_testset.epiphora_gold.isna() & epiphora_testset.epiphora_line_prediction.isna()).sum()

false_negative = ((epiphora_testset.epiphora_line == "x") & (epiphora_testset.epiphora_line_prediction.isna())).sum()

# Calculate accuracy metrics
precision = true_positive / (true_positive + false_positive)

recall = true_positive / (true_positive + false_negative)

f1_score = (2 * true_positive) / (2 * true_positive + false_positive + false_negative)

print(f"Precision: {precision * 100:2f}%")
print(f"Recall: {recall * 100:2f}%")

print(f"F1 score: {f1_score * 100:2f}%")

In [ ]:
# Error analysis: false positives

condition = epiphora_testset.epiphora_line_prediction.notna() & (
    epiphora_testset.epiphora_gold != epiphora_testset.epiphora_line_prediction
)

epiphora_testset[condition][["verse_id", "epiphora_line_prediction"]].to_csv("false_positive_epiphora_line.csv")

### Stanza epiphora

In [ ]:
## PREDICT: epiphora in stanzas
from poetry_analysis.epiphora import extract_epiphora

# group lines in the testset into stanzas
epiphora_testset[["poem_id", "stanza_id", "versenumber"]] = epiphora_testset["verse_id"].str.split("_", expand=True)

grouped_texts = epiphora_testset.groupby(["poem_id", "stanza_id"])["text"].agg(list)

results = grouped_texts.apply(extract_epiphora)

# Flatten `results` into a lookup, on the string verse_id -> overlap
pred_map = {}
for (poem_id, stanza_id), stanza_dict in results.items():  # type: ignore[union-attr]
    if not isinstance(stanza_dict, dict):
        continue
    for verse_num, payload in stanza_dict.items():
        overlap = payload.get("overlap") if isinstance(payload, dict) else None
        pred_map[f"{poem_id}_{stanza_id}_v{verse_num}"] = overlap

# Map predictions back to each original row
epiphora_testset["epiphora_stanza_prediction"] = epiphora_testset.verse_id.apply(lambda v: pred_map.get(v))

epiphora_testset = epiphora_testset.drop(["poem_id", "stanza_id", "versenumber"], axis=1)
# Optional: quick check
epiphora_testset[["verse_id", "text", "epiphora_gold", "epiphora_stanza_prediction"]].head()

In [ ]:
# EVALUATE
# Count true positives, true negatives, false positives and false negatives
true_positive = (
    (epiphora_testset.epiphora_stanza == "x")
    & (epiphora_testset.epiphora_gold == epiphora_testset.epiphora_stanza_prediction)
).sum()

false_positive = (
    epiphora_testset.epiphora_stanza_prediction.notna()
    & (epiphora_testset.epiphora_gold != epiphora_testset.epiphora_stanza_prediction)
).sum()

true_negative = (epiphora_testset.epiphora_gold.isna() & epiphora_testset.epiphora_stanza_prediction.isna()).sum()

false_negative = (
    (epiphora_testset.epiphora_stanza == "x") & (epiphora_testset.epiphora_stanza_prediction.isna())
).sum()

# Calculate accuracy metrics
precision = true_positive / (true_positive + false_positive)

recall = true_positive / (true_positive + false_negative)

f1_score = (2 * true_positive) / (2 * true_positive + false_positive + false_negative)

print(f"Precision: {precision * 100:2f}%")
print(f"Recall: {recall * 100:2f}%")

print(f"F1 score: {f1_score * 100:2f}%")

In [ ]:
# Error analysis: false positives
condition = epiphora_testset.epiphora_stanza_prediction.notna() & (
    epiphora_testset.epiphora_gold != epiphora_testset.epiphora_stanza_prediction
)

epiphora_testset[condition][["verse_id", "epiphora_stanza_prediction"]].to_csv("false_positives_epiphora_stanza.csv")

### Epiphora across stanzas 

In [ ]:
from poetry_analysis.epiphora import extract_epiphora

# group lines in the testset into lists of stanzas
stanza_texts = (
    epiphora_testset.assign(
        poem_id=epiphora_testset["verse_id"].str.extract(r"p(\d+)").astype(int),
        _stanza_num=epiphora_testset["verse_id"].str.extract(r"s(\d+)").astype(int),
        _verse_num=epiphora_testset["verse_id"].str.extract(r"v(\d+)").astype(int),
    )
    .sort_values(["poem_id", "_stanza_num", "_verse_num"])
    .groupby(["poem_id", "_stanza_num"], sort=False)["text"]
    .apply("\n".join)
)

grouped_texts = stanza_texts.groupby(level=0, sort=False).apply(list)

# extract stanza-initial lines that are repeated in several stanzas
results = grouped_texts.apply(extract_epiphora)


# Flatten `results` into a lookup, on the string verse_id -> overlap
pred_map = {}
for poem_id, stanza_dict in results.items():  # type: ignore[union-attr]
    if not isinstance(stanza_dict, dict):
        continue
    for stanza_num, payload in stanza_dict.items():
        overlap = payload.get("overlap") if isinstance(payload, dict) else None
        pred_map[f"p{poem_id}_s{stanza_num}_v0"] = overlap

# Map predictions back to each original row
epiphora_testset["epiphora_across_prediction"] = epiphora_testset.verse_id.apply(lambda v: pred_map.get(v))

# Optional: quick check
epiphora_testset[["verse_id", "text", "epiphora_gold", "epiphora_across_prediction"]].head()

In [ ]:
# EVALUATE
# Count true positives, true negatives, false positives and false negatives
true_positive = (
    (epiphora_testset.epiphora_across == "x")
    & (epiphora_testset.epiphora_gold == epiphora_testset.epiphora_across_prediction)
).sum()

false_positive = (
    epiphora_testset.epiphora_across_prediction.notna()
    & (epiphora_testset.epiphora_gold != epiphora_testset.epiphora_across_prediction)
).sum()

true_negative = (epiphora_testset.epiphora_gold.isna() & epiphora_testset.epiphora_across_prediction.isna()).sum()

false_negative = (
    (epiphora_testset.epiphora_across == "x") & (epiphora_testset.epiphora_across_prediction.isna())
).sum()

# Calculate accuracy metrics
precision = true_positive / (true_positive + false_positive)

recall = true_positive / (true_positive + false_negative)

f1_score = (2 * true_positive) / (2 * true_positive + false_positive + false_negative)

print(f"Precision: {precision * 100:2f}%")
print(f"Recall: {recall * 100:2f}%")

print(f"F1 score: {f1_score * 100:2f}%")

In [ ]:
## Error analysis: save false positives to csv
condition = epiphora_testset.epiphora_across_prediction.notna() & (
    epiphora_testset.epiphora_gold != epiphora_testset.epiphora_across_prediction
)

epiphora_testset[condition][["verse_id", "epiphora_across_prediction"]].to_csv("false_positive_epiphora_across.csv")